In [ ]:
# Turkey 타이포 수정
# 경기별 다른 가중치 적용
# 180 -> +10 상승.

import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from scipy.stats import poisson
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings

# 경고 메시지 숨기기
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# seed 고정
SEED = 9

# ==========================================
# 1. Advanced Feature Engineering (Dynamic Elo)
# ==========================================
class AdvancedElo:
    def __init__(self, k=20, base=1500, home_advantage=100):
        self.k = k
        self.ratings = {}
        self.base = base
        self.home_advantage = home_advantage

    def get_rating(self, team):
        return self.ratings.get(team, self.base)

    def expected_result(self, loc, awc):
        dr = loc - awc
        return 1 / (10 ** (-dr / 400) + 1)

    def update_rating(
        self, 
        home_team, 
        away_team, 
        home_goals, 
        away_goals, 
        neutral=False,
        tournament_weight=1.0
    ):
        home_rating = self.get_rating(home_team)
        away_rating = self.get_rating(away_team)

        goal_diff = abs(home_goals - away_goals)
        if goal_diff <= 1:
            G = 1.0
        elif goal_diff == 2:
            G = 1.5
        else:
            G = (11 + goal_diff) / 8.0

        home_adv = 0 if neutral else self.home_advantage

        home_expected = self.expected_result(home_rating + home_adv, away_rating)
        away_expected = self.expected_result(away_rating, home_rating + home_adv)

        if home_goals > away_goals:
            home_actual, away_actual = 1, 0
        elif home_goals < away_goals:
            home_actual, away_actual = 0, 1
        else:
            home_actual, away_actual = 0.5, 0.5

        home_new = home_rating + self.k * G * tournament_weight * (home_actual - home_expected)
        away_new = away_rating + self.k * G * tournament_weight * (away_actual - away_expected)

        self.ratings[home_team] = home_new
        self.ratings[away_team] = away_new

def get_tournament_weight(tournament):
    """
    경기 중요도에 따라 Elo 업데이트 강도를 조절하는 함수.
    
    - 월드컵 본선: 가장 강하게 반영
    - 대륙컵 본선: 강하게 반영
    - 월드컵 예선: 중간 이상 반영
    - 대륙컵/기타 예선: 중간 반영
    - Nations League: 약간 높게 반영
    - 친선전: 낮게 반영
    - 기타 대회: 기본값
    """
    tournament = str(tournament).strip()
    tournament_lower = tournament.lower()

    # 1. 월드컵 본선
    if tournament == "FIFA World Cup":
        return 2.0

    # 2. 월드컵 예선
    if tournament == "FIFA World Cup qualification":
        return 1.3

    # 3. 주요 대륙컵 / 국제 메이저 대회 본선
    major_tournaments = [
        "UEFA Euro",
        "Copa América",
        "African Cup of Nations",
        "AFC Asian Cup",
        "Gold Cup",
        "CONCACAF Championship",
        "Oceania Nations Cup",
        "Confederations Cup",
    ]
    if tournament in major_tournaments:
        return 1.5

    # 4. 각종 예선
    if "qualification" in tournament_lower:
        return 1.15

    # 5. Nations League 계열
    if "nations league" in tournament_lower:
        return 1.1

    # 6. 친선전
    if tournament == "Friendly":
        return 0.7

    # 7. 기타 지역컵 / 초청대회 / 소규모 대회
    return 1.0

# ==========================================
# 2. Model Training (LightGBM + Optuna Optimization)
# ==========================================
def optimize_and_train(X, y, n_trials=50):
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=SEED)
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)

    def objective(trial):
        params = {
            'objective': 'poisson',
            'metric': 'poisson',
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, 127),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'feature_pre_filter': False, 
            'verbose': -1,

            # SEED 고정
            'seed': SEED,
            'feature_fraction_seed': SEED,
            'bagging_seed': SEED,
            'data_random_seed': SEED,

            # 재현성 강화
            'deterministic': True,  
            'force_col_wise': True
        }
        
        gbm = lgb.train(
            params,
            train_data,
            valid_sets=[valid_data],
            num_boost_round=2000,
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )
        return gbm.best_score['valid_0']['poisson']

    sampler = optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(
        direction='minimize',
        sampler=sampler
    )
    
    study.optimize(objective, n_trials=n_trials)
    
    best_params = study.best_params
    best_params.update({
        'objective': 'poisson',
        'feature_pre_filter': False,
        'verbose': -1,

        'seed': SEED,
        'feature_fraction_seed': SEED,
        'bagging_seed': SEED,
        'data_random_seed': SEED,

        'deterministic': True,
        'force_col_wise': True,
    })
    
    final_model = lgb.train(
        best_params, 
        lgb.Dataset(X, label=y), 
        num_boost_round=1000
    )
    
    return final_model

def train_lgbm_models(df_train, n_trials=50):
    features = [
        'home_elo', 'away_elo', 'elo_diff', 'home_avg_scored', 'away_avg_conceded',
        'away_avg_scored', 'home_avg_conceded', 'home_form_5', 'away_form_5', 'is_neutral'
    ]
    
    X = df_train[features]
    y_home = df_train['home_score']
    y_away = df_train['away_score']

    print("\n[1/2] 홈팀 득점 예측 모델 튜닝 중...")
    lgb_home = optimize_and_train(X, y_home, n_trials=n_trials)
    
    print("\n[2/2] 원정팀 득점 예측 모델 튜닝 중...")
    lgb_away = optimize_and_train(X, y_away, n_trials=n_trials)

    return lgb_home, lgb_away, features

# ==========================================
# 3. Monte Carlo Simulator Engine
# ==========================================
def simulate_match(lambda_home, lambda_away, n_simulations=10000, random_state=SEED):
    home_goals_sim = poisson.rvs(
        mu=lambda_home, 
        size=n_simulations,
        random_state=random_state
    )
    away_goals_sim = poisson.rvs(
        mu=lambda_away, 
        size=n_simulations,
        random_state=random_state + 1
    )
    
    home_wins = np.sum(home_goals_sim > away_goals_sim)
    draws = np.sum(home_goals_sim == away_goals_sim)
    away_wins = np.sum(home_goals_sim < away_goals_sim)
    
    prob_h = home_wins / n_simulations
    prob_d = draws / n_simulations
    prob_a = away_wins / n_simulations
    
    scores = [f"{h}-{a}" for h, a in zip(home_goals_sim, away_goals_sim)]
    most_frequent_score = max(set(scores), key=scores.count)
    pred_h_score, pred_a_score = map(int, most_frequent_score.split('-'))
    
    return prob_h, prob_d, prob_a, pred_h_score, pred_a_score

def predict_tournament(test_matches, lgb_home, lgb_away, features, n_simulations=10000):
    X_test = test_matches[features]
    
    test_matches['lambda_home'] = lgb_home.predict(X_test)
    test_matches['lambda_away'] = lgb_away.predict(X_test)
    
    results = []
    print(f"\n🏃‍♂️ 각 경기당 {n_simulations:,}번의 몬테카를로 시뮬레이션을 실행합니다...")
    
    for idx, row in tqdm(test_matches.iterrows(), total=len(test_matches)):
        p_home, p_draw, p_away, pred_h, pred_a = simulate_match(row['lambda_home'], row['lambda_away'], n_simulations)
        
        # ---------------------------------------------------------
        # 💡 [유저 특별 요청 반영] 무승부 확률 비례 분배 (강팀 어드밴티지)
        # ---------------------------------------------------------
        if (p_home + p_away) > 0:
            # 기본 승리 확률의 비율만큼 무승부 확률을 나눠 갖습니다.
            team1_prob_final = p_home + p_draw * (p_home / (p_home + p_away))
            team2_prob_final = p_away + p_draw * (p_away / (p_home + p_away))
        else:
            team1_prob_final = 0.5
            team2_prob_final = 0.5
            
        # ---------------------------------------------------------
        # 💡 [스코어 타이브레이커] 무승부 점수가 나왔을 경우 승패 강제 결정
        # ---------------------------------------------------------
        if pred_h == pred_a:
            if team1_prob_final > team2_prob_final:
                pred_h += 1 # 1점 더 높은 팀이 연장전 끝에 승리했다고 간주
            elif team2_prob_final > team1_prob_final:
                pred_a += 1

        match_type = row.get('type', 'group')
        if pd.isna(match_type) or match_type == '':
            match_type = 'group'

        results.append({
            'team1': row['home_team'],
            'team2': row['away_team'],
            'team1_score': int(pred_h),
            'team2_score': int(pred_a),
            'team1_prob': round(team1_prob_final, 4),
            'team2_prob': round(team2_prob_final, 4),
            'type': match_type
        })
        
    output_cols = ['team1', 'team2', 'team1_score', 'team2_score', 'team1_prob', 'team2_prob', 'type']
    return pd.DataFrame(results, columns=output_cols)

def format_submission(df):
    """
    submission_consensus.csv를 채점 가능한 제출 양식으로 보정하는 함수
    - 팀명 표준화
    - type 컬럼 표준화
    - 컬럼 순서 고정
    - 점수/확률 타입 정리
    """

    df = df.copy()

    # 1. 팀 이름 표준화
    name_corrections = {
        "Cura?ao": "Curaçao",
        "Curacao": "Curaçao",
        "DR Congo": "Congo DR",
        "Cape Verde": "Cape Verde Islands",
        "Czech Republic": "Czechia",
        "Bosnia and Herzegovina": "Bosnia-Herzegovina"
    }

    df["team1"] = df["team1"].replace(name_corrections)
    df["team2"] = df["team2"].replace(name_corrections)

    # 2. type 컬럼 형식 변경
    if "type" in df.columns:
        df["type"] = (
            df["type"]
            .astype(str)
            .str.strip()
            .str.lower()
            .replace({
                "group": "Group Stage",
                "group stage": "Group Stage",
            })
        )

    # 3. 컬럼 순서 고정
    desired_columns = [
        "team1",
        "team2",
        "team1_score",
        "team2_score",
        "team1_prob",
        "team2_prob",
        "type",
    ]

    df = df[desired_columns].copy()

    # 4. 점수 컬럼 정수형 보정
    df["team1_score"] = pd.to_numeric(df["team1_score"], errors="coerce").fillna(0).astype(int)
    df["team2_score"] = pd.to_numeric(df["team2_score"], errors="coerce").fillna(0).astype(int)

    # 5. 확률 컬럼 숫자형 보정
    df["team1_prob"] = pd.to_numeric(df["team1_prob"], errors="coerce").fillna(0.5)
    df["team2_prob"] = pd.to_numeric(df["team2_prob"], errors="coerce").fillna(0.5)

    # 6. 확률 합 1로 정규화
    prob_sum = df["team1_prob"] + df["team2_prob"]

    bad_mask = prob_sum <= 0
    df.loc[bad_mask, "team1_prob"] = 0.5
    df.loc[bad_mask, "team2_prob"] = 0.5

    prob_sum = df["team1_prob"] + df["team2_prob"]

    df["team1_prob"] = (df["team1_prob"] / prob_sum).round(4)
    df["team2_prob"] = (1.0 - df["team1_prob"]).round(4)

    return df

# ==========================================
# 4. Execution Pipeline (캐글 실행부)
# ==========================================
if __name__ == "__main__":
    print("🚀 월드컵 대진표 예측 파이프라인 시작...\n")
    
    print("데이터 로드 및 Elo 레이팅, 스탯 계산 중...")
    file_path = "../data/historical_results.csv"
    df = pd.read_csv(file_path)
    
    df['home_score'] = pd.to_numeric(df['home_score'], errors='coerce')
    df['away_score'] = pd.to_numeric(df['away_score'], errors='coerce')

    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    # df['is_neutral'] = df['neutral'].apply(lambda x: 1 if str(x).upper() == 'TRUE' else 0)
    df["is_neutral"] = (
        df["neutral"]
        .astype(str)
        .str.strip()
        .str.upper()
        .isin(["TRUE", "1", "YES", "Y"])
        .astype(int)
    )

    df["tournament_weight"] = df["tournament"].apply(get_tournament_weight)


    elo_system = AdvancedElo()
    home_elos, away_elos = [], []
    team_stats = {}
    h_avg_s, h_avg_c, a_avg_s, a_avg_c, h_form, a_form = [], [], [], [], [], []

    for idx, row in df.iterrows():
        ht, at = row['home_team'], row['away_team']
        
        home_elos.append(elo_system.get_rating(ht))
        away_elos.append(elo_system.get_rating(at))
        
        for t in (ht, at):
            if t not in team_stats:
                team_stats[t] = {'scored': [], 'conceded': [], 'pts': []}
                
        if len(team_stats[ht]['scored']) > 0:
            h_avg_s.append(np.mean(team_stats[ht]['scored'][-10:]))
            h_avg_c.append(np.mean(team_stats[ht]['conceded'][-10:]))
            h_form.append(sum(team_stats[ht]['pts'][-5:]))
        else:
            h_avg_s.append(1.0); h_avg_c.append(1.0); h_form.append(0)
            
        if len(team_stats[at]['scored']) > 0:
            a_avg_s.append(np.mean(team_stats[at]['scored'][-10:]))
            a_avg_c.append(np.mean(team_stats[at]['conceded'][-10:]))
            a_form.append(sum(team_stats[at]['pts'][-5:]))
        else:
            a_avg_s.append(1.0); a_avg_c.append(1.0); a_form.append(0)

        if pd.notna(row['home_score']) and pd.notna(row['away_score']):
            hs, ast = float(row['home_score']), float(row['away_score'])
            
            # elo_system.update_rating(
            #     home_team=ht, 
            #     away_team=at, 
            #     home_goals=hs, 
            #     away_goals=ast,
            #     neutral=bool(row["is_neutral"])
            # )
            elo_system.update_rating(
                home_team=ht, 
                away_team=at, 
                home_goals=hs, 
                away_goals=ast,
                neutral=bool(row["is_neutral"]),
                tournament_weight=row["tournament_weight"]
            )
            
            team_stats[ht]['scored'].append(hs)
            team_stats[ht]['conceded'].append(ast)
            team_stats[at]['scored'].append(ast)
            team_stats[at]['conceded'].append(hs)
            
            if hs > ast:
                team_stats[ht]['pts'].append(3); team_stats[at]['pts'].append(0)
            elif hs < ast:
                team_stats[ht]['pts'].append(0); team_stats[at]['pts'].append(3)
            else:
                team_stats[ht]['pts'].append(1); team_stats[at]['pts'].append(1)

    df['home_elo'] = home_elos
    df['away_elo'] = away_elos
    df['elo_diff'] = df['home_elo'] - df['away_elo']
    df['home_avg_scored'] = h_avg_s
    df['home_avg_conceded'] = h_avg_c
    df['away_avg_scored'] = a_avg_s
    df['away_avg_conceded'] = a_avg_c
    df['home_form_5'] = h_form
    df['away_form_5'] = a_form

    mask_2026_wc = (df['date'].dt.year == 2026) & (df['tournament'] == 'FIFA World Cup')
    
    df_train = df[~mask_2026_wc].dropna(subset=['home_score', 'away_score']).copy()
    df_test = df[mask_2026_wc].copy()
    
    print(f"\n✅ 피처 계산 완료! 훈련 데이터: {len(df_train):,}건 / 타겟 경기: {len(df_test):,}건\n")

    # ⚠️ 제출 시에는 OPTUNA_TRIALS=50, MONTE_CARLO_SIMS=10000 권장
    OPTUNA_TRIALS = 50 
    lgb_home, lgb_away, feature_cols = train_lgbm_models(df_train, n_trials=OPTUNA_TRIALS)

    MONTE_CARLO_SIMS = 10000 
    df_predictions = predict_tournament(df_test, lgb_home, lgb_away, feature_cols, n_simulations=MONTE_CARLO_SIMS)
    df_predictions = format_submission(df_predictions)

    output_filename = "submission_consensus.csv"
    df_predictions.to_csv(output_filename, index=False)
    
    print("\n==========================================")
    print(f"🎉 예측 완료! 캐글 형식에 맞춘 결과가 '{output_filename}'에 저장되었습니다.")
    print("==========================================")
    print(df_predictions.head())

/home/brik0519/anaconda3/envs/facamp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 월드컵 대진표 예측 파이프라인 시작...

데이터 로드 및 Elo 레이팅, 스탯 계산 중...

✅ 피처 계산 완료! 훈련 데이터: 49,405건 / 타겟 경기: 72건


[1/2] 홈팀 득점 예측 모델 튜닝 중...

[2/2] 원정팀 득점 예측 모델 튜닝 중...

🏃‍♂️ 각 경기당 10,000번의 몬테카를로 시뮬레이션을 실행합니다...


100%|██████████| 72/72 [00:01<00:00, 50.31it/s]



🎉 예측 완료! 캐글 형식에 맞춘 결과가 'submission_consensus.csv'에 저장되었습니다.
           team1               team2  team1_score  team2_score  team1_prob  \
0    South Korea             Czechia            2            1      0.6327   
1         Mexico        South Africa            2            0      0.9048   
2         Canada  Bosnia-Herzegovina            2            0      0.8864   
3  United States            Paraguay            2            1      0.6451   
4          Qatar         Switzerland            0            2      0.1058   

   team2_prob         type  
0      0.3673  Group Stage  
1      0.0952  Group Stage  
2      0.1136  Group Stage  
3      0.3549  Group Stage  
4      0.8942  Group Stage  
